# Tidal composites generation using odc-stats

Apply geomedian to Sentinel-2 imagery at the highest and lower tide ranges.

This notebook uses odc-stats to generate the tidal composites.

## Construct tasks

Test using 10km tiles.

In [ ]:
import json

def save_tasks(cloud_cover, output_db):
    dataset_filter = json.dumps({"cloud_cover": [0, cloud_cover]})
    
    !uv run odc-stats save-tasks \
        --frequency "rolling-3years" \
        --grid "EPSG:6933;10;1000" \
        --temporal-range 2023--P3Y \
        --input-products "s2_l2a" \
        --dataset-filter='{{"cloud_cover": [0,{cloud_cover}]}}' \
        {output_db}
    
save_tasks(20, "rolling_3year_10km_20.db")

In [ ]:
import geopandas as gpd

gdf = gpd.read_file("rolling_3year_10km_20-2023--P3Y.geojson")
#gdf.explore()

In [ ]:
# filter to coastal tiles
coast_gdf = gpd.read_file("https://raw.githubusercontent.com/piksel-ina/Indonesia-coastlines/main/data/raw/indonesia_10km_tiles_coast.geojson")
gdf_filtered = gdf[gdf.intersects(coast_gdf.to_crs(gdf.crs).union_all())]

In [ ]:
gdf_filtered.explore()

### Examine the tasks

In [ ]:
from odc.stats.tasks import TaskReader
from odc.stats.model import OutputProduct

## Open the task database to find our tile, we need to create the OutputProduct class
#  to open the taskdb but it doesn't do anything.
op = OutputProduct(
            name="hltc_s2",
            version="0.0.1",
            short_name="hltc_s2",
            location=f"s3://dummy-bucket/", #this is a fake path
            properties={"odc:file_format": "GeoTIFF"},
            measurements=['red'], #any measurements, doesn't matter.
        )

taskdb = TaskReader(f'rolling_3year_10km_20.db', product=op)

### Load task for a selected tile

In [ ]:
#t = 230,-15 #50km tile
t=1153,-72 #10km tile
#select our individual task i.e. our tile
task = taskdb.load_task((f'2023--P3Y', t[0], t[1]))
task

In [ ]:
len(task.datasets)

## Run a task

In [ ]:
from odc.stats._text import parse_yaml_file_or_inline

main_config="/home/jovyan/dev/Indonesia-GeoMAD/config/hltc_config/main_config.yaml"
plugin_config="/home/jovyan/dev/Indonesia-GeoMAD/config/hltc_config/plugin_config.yaml"

cfg = parse_yaml_file_or_inline(plugin_config)
cfg

In [ ]:
from odc.stats.plugins import resolve
mk_proc = resolve("hltc.HLTidalComposites")
proc = mk_proc(**cfg)
proc

In [ ]:
# run the separate functions
tc = proc.input_data(datasets=task.datasets, geobox=task.geobox)

In [ ]:
tc

In [ ]:
ds_tidalcomposites = proc.reduce(tc).compute()
ds_tidalcomposites

In [ ]:
ds_tidalcomposites[["low_red", "low_green", "low_blue"]].plot.imshow(robust=True)

## Run a task in command line

In [ ]:
!uv run odc-stats run rolling_3year_10km_20.db 2023--P3Y/1153/-72 --plugin="hltc.HLTidalComposites" \
--config="/home/jovyan/dev/Indonesia-GeoMAD/config/hltc_config/main_config.yaml" \
--plugin-config="/home/jovyan/dev/Indonesia-GeoMAD/config/hltc_config/plugin_config.yaml" \
--threads="4" \
--memory-limit="24GB" \
--location="file:///home/jovyan/dev/Indonesia-GeoMAD/notebook/output/"

## Check output

In [ ]:
import json
import pystac
import odc.stac

stac = "output/x1153/y-072/2023--P3Y/htlc_s2_x1153y-072_2023--P3Y.stac-item.json"

# Load STAC item JSON
with open(stac) as f:
    item_dict = json.load(f)

item = pystac.Item.from_dict(item_dict)

# Load into xarray
ds = odc.stac.load(
    [item],
    bands=[
        "low_red",
        "low_green",
        "low_blue",
        "low_nir",
        "high_red",
        "high_green",
        "high_blue",
        "high_nir",
        "qa_count_clear_low",
        "qa_count_clear_high",
        "qa_count_clear_total",
        "qa_low_threshold",
    ],
).squeeze()

ds

In [ ]:
ds['ndwi_low'] = (ds['low_green']-ds['low_nir'])/(ds['low_green']+ds['low_nir'])
ds['ndwi_high']= (ds['high_green']-ds['high_nir'])/(ds['high_green']+ds['high_nir'])

In [ ]:
# intertidal area
((ds['ndwi_low']>0)*1.0 - (ds['ndwi_high']>0)*1.0).plot.imshow(robust=True);